# Prophet Pipeline: Parameters and Execution
This notebook is used to:
- configure input/output paths together with PSO and Prophet parameters
- call `run_prophet_pipeline` from `functions/prophet_pipeline.py` to fit and fill the time series
- optionally enable plotting and coordinate-field output

In [ ]:
import sys, os
from pprint import pprint
import importlib

# __file__ is not available in notebooks; infer the project root from the current working directory
cwd = os.getcwd()
workspace_root = os.path.dirname(cwd) if os.path.basename(cwd) == 'ntl_prophet' else cwd
package_root = os.path.join(workspace_root, 'ntl_prophet')
if workspace_root not in sys.path:
    sys.path.insert(0, workspace_root)
if package_root not in sys.path:
    sys.path.insert(0, package_root)

from ntl_prophet.functions import run_prophet_pipeline
import ntl_prophet.functions.prophet_pipeline as _pp
_pp = importlib.reload(_pp)
run_prophet_pipeline = _pp.run_prophet_pipeline

In [ ]:
# 1) Input / output
INPUT_DIRS = [r'.\output\ntl_timeseries_angle.txt']
OUTPUT_DIRS = [r'.\output\ntl_timeseries_angle_prophet_pso.txt']

# 2) Time-window settings (three-digit Julian day)
DOY_START = '001'
DOY_END = '120'

# 3) Filtering conditions
MIN_OBS_PER_POINT = 10  # Minimum number of observations required per pixel

# 4) PSO parameters
PSO_CONFIG = {
    'particle_num': 3,
    'particle_dim': 3,
    'iter_num': 3,
    'c1': 2.0,
    'c2': 2.0,
    'w': 0.5,
    'min_value': 0.01,
    'max_n_changepoints_ratio': 0.6,
}

# 5) Fixed Prophet parameters
PROPHET_CONFIG = {
    'seasonality_mode': 'additive',
    'daily_seasonality': False,
    'weekly_seasonality': False,
    'changepoint_range': 1,
}

# 6) Other settings
FUTURE_DAYS = 0
FLAGS = {
    'enable_pso_plot': False,
    'enable_forecast_plot': False,
    'enable_simple_series_plot': False,
    'output_include_coords': False,
    'progress_bar': True,
    'quiet': True,
    'parallel_points': True,
    'num_workers': 16,
}
RANDOM_SEED = 42

In [ ]:
# Run the pipeline
summary = run_prophet_pipeline(
    input_dirs=INPUT_DIRS,
    output_dirs=OUTPUT_DIRS,
    loop_max_freq=1,
    doy_start=DOY_START,
    doy_end=DOY_END,
    min_obs_per_point=MIN_OBS_PER_POINT,
    pso_config=PSO_CONFIG,
    prophet_config=PROPHET_CONFIG,
    future_days=FUTURE_DAYS,
    flags=FLAGS,
    random_seed=RANDOM_SEED,
)
pprint(summary)

In [ ]:
# Inspect the saved accuracy report
import os

metrics_list = summary.get('metrics', []) if isinstance(summary, dict) else []
print('=== Metrics summary by file ===')
for m in metrics_list:
    if not m:
        continue
    print('file:', m.get('file'))
    print('points:', m.get('points'))
    print('mae_avg:', m.get('mae_avg'), 'mae_med:', m.get('mae_med'))
    print('rel_avg:', m.get('rel_avg'), 'rel_med:', m.get('rel_med'))
    if m.get('report_csv'):
        print('report_csv:', m.get('report_csv'))
    print('-')

for m in metrics_list:
    report = m.get('report_csv') if isinstance(m, dict) else None
    if report and os.path.exists(report):
        print(f'Preview of {report}:')
        try:
            with open(report, 'r', encoding='utf-8') as rf:
                for i, line in enumerate(rf):
                    print(line.rstrip())
                    if i >= 10:
                        break
        except Exception as e:
            print('(cannot read report)', e)

In [ ]:
# Convert txt output to daily GeoTIFFs
import os
from functions.text_to_img import txt_to_daily_geotiffs

TXT_PATH = OUTPUT_DIRS[0]
IMG_OUTPUT_DIR = r'.\output\prophet_daily_tif'
START_DATE = '20180101'
END_DATE = '20190101'

# Option A: use a template raster (recommended for reliable spatial metadata)
TEMPLATE_TIF = r'.\template.tif'

if os.path.exists(TEMPLATE_TIF):
    tif_files = txt_to_daily_geotiffs(
        txt_path=TXT_PATH,
        output_dir=IMG_OUTPUT_DIR,
        start_date=START_DATE,
        end_date=END_DATE,
        template_tif=TEMPLATE_TIF,
    )
else:
    tif_files = txt_to_daily_geotiffs(
        txt_path=TXT_PATH,
        output_dir=IMG_OUTPUT_DIR,
        start_date=START_DATE,
        end_date=END_DATE,
        template_tif=None,
        width=1000,
        height=1000,
        transform=(0.0041666667, 0.0, 70.0, 0.0, -0.0041666667, 55.0),
        crs='EPSG:4326',
    )

print('Number of generated images:', len(tif_files))
print('First file:', tif_files[0] if tif_files else '(none)')